# Initialize — TPC-H Sample (single catalog)

Shared variables + helpers for the staging notebooks. Invoke via `%run ./initialize`.
Every medallion layer lives in ONE catalog, separated by schema
(`{schema_namespace}_<layer>[_<source>]{logical_env}`), derived from `catalog`,
`schema_namespace` and `logical_env`.

In [ ]:
from datetime import datetime, timedelta
import pyspark.sql.functions as F

dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema_namespace", "tpch_sample")
dbutils.widgets.text("logical_env", "")

catalog = dbutils.widgets.get("catalog")
schema_namespace = dbutils.widgets.get("schema_namespace")
logical_env = dbutils.widgets.get("logical_env")

# Single-catalog model: one catalog, one schema per medallion layer (and per bronze source)
staging_schema = f"{schema_namespace}_staging{logical_env}"
silver_schema = f"{schema_namespace}_silver{logical_env}"
gold_schema = f"{schema_namespace}_gold{logical_env}"
staging_volume = "stg_volume"

# Per-source-system bronze schemas
bronze_source_systems = [
    "reference_data", "crm", "procurement", "vendor_mgmt",
    "order_mgmt", "order_fulfillment", "product_catalog", "inventory",
]


def bronze_schema(system):
    return f"{schema_namespace}_bronze_{system}{logical_env}"


sample_source_schema = "samples.tpch"

volume_root = f"/Volumes/{catalog}/{staging_schema}/{staging_volume}"
sources_root = f"{volume_root}/sources"

# Staging uses Parquet (typed, self-describing) so bronze can infer + evolve the schema
# via Auto Loader. No source schema is declared in the bronze flows.

# Landing-zone layout: sources/{system}/{entity}/{zone}/{entity}_<yyyymmddHHMMSS>.parquet
# zone "initial"     -> day-1 baseline (setup job), persists across full refreshes
# zone "incremental" -> day-2/3 batches (run_2/run_3), safe to wipe via reset_incremental()
INITIAL_ZONE = "initial"
INCREMENTAL_ZONE = "incremental"


def landing_dir(system, entity, zone):
    return f"{sources_root}/{system}/{entity}/{zone}"


def write_landing(df, system, entity, zone, load_ts):
    """Write df as a single timestamped Parquet file ({entity}_<ts>.parquet) into the zone
    folder, mimicking a real landing zone. The filename timestamp matches the batch load_ts."""
    suffix = load_ts.strftime("%Y%m%d%H%M%S")
    target_dir = landing_dir(system, entity, zone)
    fname = f"{entity}_{suffix}.parquet"
    tmp_dir = f"{target_dir}/_tmp_{fname}"
    df.coalesce(1).write.format("parquet").mode("overwrite").save(tmp_dir)
    part = [f.path for f in dbutils.fs.ls(tmp_dir) if f.name.endswith(".parquet")][0]
    dbutils.fs.mv(part, f"{target_dir}/{fname}")
    dbutils.fs.rm(tmp_dir, True)
    print(f"  wrote {system}/{entity}/{zone}/{fname}")


def reset_incremental():
    """Remove every incremental/ landing folder so a full refresh reprocesses the
    day-1 initial/ data only (no need to re-stage the initial sample)."""
    reset = 0
    for system in bronze_source_systems:
        try:
            entities = dbutils.fs.ls(f"{sources_root}/{system}")
        except Exception:
            continue
        for e in entities:
            inc = f"{e.path.rstrip('/')}/{INCREMENTAL_ZONE}"
            try:
                dbutils.fs.rm(inc, True)
                reset += 1
            except Exception:
                pass
    print(f"Reset incremental landing zone ({reset} entity folders cleared).")


def with_metadata(df, system, batch_id, load_ts):
    return (df
            .withColumn("load_timestamp", F.lit(load_ts).cast("timestamp"))
            .withColumn("source_system", F.lit(system))
            .withColumn("batch_id", F.lit(batch_id)))


def with_effective(df, effective_date):
    """Stamp a business effective date (the SCD2 sequence_by for the customer/supplier
    dimensions). This is on the business timeline (1992-1998) so gold facts can resolve the
    dimension version that was effective as of the order_date (point-in-time). Distinct from
    load_timestamp, which is processing time."""
    return df.withColumn("effective_date", F.lit(effective_date).cast("timestamp"))


def with_op(df, op="U"):
    """Stamp a CDC operation flag (cdc_operation) carried by every SCD2 dimension feed. 'U' is a
    normal upsert (insert/update); 'D' marks a delete. Silver applies it via
    cdcSettings.apply_as_deletes ("cdc_operation = 'D'") so a delete closes the open SCD2 version
    (tombstone) instead of inserting. The column is in except_column_list, so it drives the
    merge but is never stored in the target."""
    return df.withColumn("cdc_operation", F.lit(op))


print("Initialized:")
print(f"  catalog        = {catalog}")
print(f"  staging_schema = {staging_schema}")
print(f"  silver_schema  = {silver_schema}")
print(f"  gold_schema    = {gold_schema}")
print(f"  sources_root   = {sources_root}")